In [3]:
# """
# 從 Trello API 取得資料
# ----------------------
# 
# ⚠️ 注意：這個 notebook 需要 Trello API 金鑰才能執行
# 
# 如果你沒有 Trello 帳號或 API 金鑰，可以：
# 1. 跳過這個 notebook
# 2. 直接使用 sample_data/訂單資料_範例.csv 進行後續分析
# 
# 如果你想要用自己的 Trello 資料，請先：
# 1. 申請 API 金鑰 (https://trello.com/power-ups/admin)
# 2. 複製 .env.example 為 .env 並填入你的金鑰
# 3. 取得你的看板 ID（從 Trello 網址）
# """
import requests
import json
import os
import re
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd
import traceback
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time

# 加载环境变量
load_dotenv(r"C:\Users\MI\Desktop\desktop\trello_project\trello_data.env")
API_KEY = os.getenv("TRELLO_API_KEY")
TOKEN = os.getenv("TRELLO_TOKEN")

# 測試 API Key 和 Token 是否有效
def test_api_credentials():
    """測試 API Key 和 Token 是否有效"""
    print("🔐 測試 API 憑證...")
    test_url = "https://api.trello.com/1/members/me"
    params = {
        "key": API_KEY,
        "token": TOKEN
    }
    
    try:
        response = requests.get(test_url, params=params)
        if response.status_code == 200:
            user_data = response.json()
            print(f"✅ API 憑證有效！用戶: {user_data.get('fullName', '未知')}")
            print(f"   用戶名: {user_data.get('username', '未知')}")
            return True
        else:
            print(f"❌ API 憑證無效！狀態碼: {response.status_code}")
            print(f"   錯誤訊息: {response.text}")
            return False
    except Exception as e:
        print(f"❌ API 測試失敗: {str(e)}")
        return False

class TrelloOrderAnalyzer:
    def __init__(self, board_id):
        self.base_url = "https://api.trello.com/1"
        self.params = {"key": API_KEY, "token": TOKEN}
        self.board_id = board_id
        self.custom_fields_map = {}
        self.field_options = {}
        self.selenium_driver = None
        
    def init_selenium(self):
        """初始化Selenium浏览器"""
        try:
            print("🚀 正在启动Selenium浏览器...")
            chrome_options = Options()
            chrome_options.add_argument("--headless")  # 无头模式
            chrome_options.add_argument("--disable-gpu")
            chrome_options.add_argument("--no-sandbox")
            chrome_options.add_argument("--disable-dev-shm-usage")
            chrome_options.add_argument("--window-size=1920,1080")
            
            # 使用 Service 和 options 分开传递
            service = Service(ChromeDriverManager().install())
            self.selenium_driver = webdriver.Chrome(
                service=service,
                options=chrome_options
            )
            self.selenium_driver.set_page_load_timeout(30)
            print("✅ Selenium浏览器启动成功")
            return True
        except Exception as e:
            print(f"❌ 启动Selenium浏览器失败: {str(e)}")
            return False
    
    def get_custom_fields(self):
        """获取自定义字段定义"""
        try:
            print("🔍 获取自定义字段...")
            url = f"{self.base_url}/boards/{self.board_id}/customFields"
            response = requests.get(url, params=self.params)
                
            if response.status_code != 200:
                print(f"⚠️ 获取自定义字段失败，状态码: {response.status_code}")
                print(f"   错误信息: {response.text}")
                return {}
                
            fields = response.json()
            for field in fields:
                field_id = field["id"]
                field_name = field["name"]
                self.custom_fields_map[field_id] = field_name
                
                # 存储下拉选项
                if field["type"] == "list" and field.get("options"):
                    self.field_options[field_id] = {
                        opt["id"]: opt["value"]["text"] for opt in field["options"]
                    }
                    print(f"   ✅ 字段 '{field_name}' 找到 {len(field['options'])} 个选项")
            
            print(f"✅ 共找到 {len(self.custom_fields_map)} 个自定义字段")
            return self.custom_fields_map
            
        except Exception as e:
            print(f"❌ 获取自定义字段时出错: {str(e)}")
            return {}
    
    def clean_value(self, value):
        """数据清洗：处理特殊字符和格式"""
        if value is None:
            return ""
            
        if isinstance(value, list):
            return ",".join([self.clean_value(v) for v in value])
        
        if isinstance(value, dict):
            # 处理复杂对象
            if "text" in value:
                return self.clean_value(value["text"])
            if "value" in value:
                return self.clean_value(value["value"])
            return json.dumps(value)
        
        if isinstance(value, (int, float)):
            return value
        
        # 字符串处理
        value = str(value)
        # 替换换行符
        value = value.replace('\n', ' ').replace('\r', ' ')
        # 保留中文、英文、数字和常见标点
        value = re.sub(r'[^\w\s\u4e00-\u9fff\u3400-\u4dbf\.,，。！？：；、\-\+\(\)（）]', '', value)
            
        return value.strip()
    
    def extract_card_data(self, card):
        """从卡片提取订单数据"""
        try:
            # 初始化所有字段
            order_data = {
                "Card_ID": card.get("id", ""),
                "Card_Name": card.get("name", ""),
                "List_ID": card.get("idList", ""),
                "Date": None,
                "Product_ID": None,
                "Total_amount": None,
                "Phone": None
            }
            
            # 处理自定义字段
            for field_item in card.get("customFieldItems", []):
                try:
                    field_id = field_item["idCustomField"]
                    field_name = self.custom_fields_map.get(field_id)
                    
                    if not field_name:
                        continue
                        
                    # 只处理需要的字段
                    if field_name not in ["Date", "Product_ID", "Total_amount", "Phone"]:
                        continue
                        
                    value_data = field_item.get("value")
                    
                    # 处理各种字段类型
                    if isinstance(value_data, dict):
                        # 下拉字段选项
                        if "id" in value_data:
                            option_id = value_data["id"]
                            options = self.field_options.get(field_id, {})
                            value = options.get(option_id, "")
                        # 文本字段
                        elif "text" in value_data:
                            value = value_data["text"]
                        # 数字字段
                        elif "number" in value_data:
                            try:
                                value = float(value_data["number"])
                            except (TypeError, ValueError):
                                value = value_data["number"]
                        # 日期字段
                        elif "date" in value_data:
                            value = value_data["date"].split("T")[0] if value_data["date"] else ""
                        else:
                            value = ""
                    else:
                        value = value_data
                    
                    # 应用数据清洗
                    cleaned_value = self.clean_value(value)
                    order_data[field_name] = cleaned_value
                    
                except Exception as e:
                    print(f"⚠️ 处理字段 {field_name} 时出错: {str(e)}")
                    continue
            
            # 使用截止日期作为备选Date
            if card.get("due") and not order_data["Date"]:
                order_data["Date"] = card["due"].split("T")[0] if card["due"] else ""
            
            return order_data
            
        except Exception as e:
            print(f"❌ 提取卡片数据出错: {str(e)}")
            return {
                "Card_ID": card.get("id", "unknown"),
                "Card_Name": card.get("name", "unknown"),
                "List_ID": card.get("idList", "unknown"),
                "Date": None,
                "Product_ID": None,
                "Total_amount": None,
                "Phone": None
            }
    
    def get_board_lists(self):
        """获取看板所有列表"""
        try:
            print("🔍 获取看板列表...")
            url = f"{self.base_url}/boards/{self.board_id}/lists"
            params = {
                **self.params,
                "fields": "id,name"
            }
            
            response = requests.get(url, params=params)
            if response.status_code == 200:
                lists = response.json()
                list_dict = {lst["id"]: lst["name"] for lst in lists}
                print(f"✅ 找到 {len(list_dict)} 个列表")
                return list_dict
            else:
                print(f"⚠️ 获取看板列表失败，状态码: {response.status_code}")
                return {}
        except Exception as e:
            print(f"❌ 获取看板列表时出错: {str(e)}")
            return {}
    
    def get_board_members(self):
        """获取看板成员"""
        try:
            url = f"{self.base_url}/boards/{self.board_id}/members"
            response = requests.get(url, params=self.params)
            if response.status_code == 200:
                members = response.json()
                return {m["id"]: m.get("fullName", m.get("username", "未知")) for m in members}
            return {}
        except:
            return {}
    
    def get_all_cards(self):
        """获取看板所有卡片"""
        try:
            print("🔍 获取看板卡片...")
            url = f"{self.base_url}/boards/{self.board_id}/cards"
            params = {
                **self.params,
                "customFieldItems": "true",
                "fields": "name,due,desc,idList"
            }
            
            response = requests.get(url, params=params)
            if response.status_code == 200:
                cards = response.json()
                print(f"✅ 找到 {len(cards)} 张卡片")
                return cards
            else:
                print(f"❌ 获取卡片失败，状态码: {response.status_code}")
                print(f"   错误信息: {response.text}")
                return []
                
        except Exception as e:
            print(f"❌ 获取卡片时出错: {str(e)}")
            return []
    
    def export_order_report(self):
        """生成订单报表"""
        print("\n" + "="*50)
        print("开始导出订单数据...")
        
        # 1. 获取自定义字段
        self.get_custom_fields()
        
        # 2. 获取列表信息
        lists = self.get_board_lists()
        
        # 3. 获取所有卡片
        cards = self.get_all_cards()
        
        if not cards:
            print("❌ 没有找到任何卡片，请检查：")
            print("   1. API Key 和 Token 是否正确")
            print("   2. 看板ID是否正确")
            print("   3. 您是否有权限访问这个看板")
            return None, None
        
        orders = []
        print(f"\n📋 处理 {len(cards)} 张卡片...")
        
        for i, card in enumerate(cards):
            try:
                order_data = self.extract_card_data(card)
                
                # 添加列表名称
                list_id = order_data.get("List_ID", "")
                order_data["List_Name"] = lists.get(list_id, "未知列表")
                
                orders.append(order_data)
                
                if (i+1) % 10 == 0 or (i+1) == len(cards):
                    print(f"⏳ 已处理 {i+1}/{len(cards)} 张卡片")
                    
            except Exception as e:
                print(f"❌ 处理卡片 {i+1} 时出错: {str(e)}")
                continue
        
        # 创建DataFrame
        df = pd.DataFrame(orders)
        
        # 定义需要的字段顺序
        required_fields = [
            "Card_ID", 
            "Card_Name", 
            "List_ID", 
            "List_Name", 
            "Date", 
            "Product_ID", 
            "Total_amount", 
            "Phone"
        ]
        
        # 确保所有字段都存在
        for field in required_fields:
            if field not in df.columns:
                df[field] = ""
        
        # 按顺序排列字段
        df = df[required_fields]
        
        # 替换NaN值为空字符串
        df = df.fillna("")
        
        # 过滤掉完全空的行（除了卡片ID和名称之外的所有字段都为空）
        mask = (df["Date"] != "") | (df["Product_ID"] != "") | (df["Total_amount"] != "") | (df["Phone"] != "")
        df = df[mask]
        
        if len(df) == 0:
            print("⚠️ 警告：没有提取到任何有效数据")
            print("   请检查：")
            print("   1. 卡片是否有自定义字段")
            print("   2. 自定义字段名称是否正确")
        else:
            print(f"✅ 成功提取 {len(df)} 笔订单数据")
        
        # 保存CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"订单明细报表_{timestamp}.csv"
        df.to_csv(filename, index=False, encoding="utf-8-sig")
        
        return filename, df

def get_board_id_from_url(board_url):
    """从Trello看板URL中提取ID"""
    patterns = [
        r"https://trello\.com/b/([a-zA-Z0-9]+)/",
        r"b/([a-zA-Z0-9]{8})",
        r"https://trello\.com/b/([a-zA-Z0-9]+)$"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, board_url)
        if match:
            return match.group(1)
    
    return None

if __name__ == "__main__":
    print("="*50)
    print("Trello 订单数据导出工具")
    print("="*50)
    
    # 测试 API 凭据
    if not test_api_credentials():
        print("\n❌ API 凭据无效，请检查 .env 文件")
        print("   确保已正确设置：")
        print(f"   1. TRELLO_API_KEY = {API_KEY}")
        print(f"   2. TRELLO_TOKEN = {TOKEN}")
        exit(1)
    
    # 获取看板ID
    BOARD_ID = "Nr2kR7ni"  # 直接使用您的看板ID
    
    # 或者手动输入
    if not BOARD_ID:
        board_url = input("\n🔍 请输入Trello看板URL: ").strip()
        BOARD_ID = get_board_id_from_url(board_url)
        
        if not BOARD_ID:
            BOARD_ID = input("🔍 请直接输入看板ID: ").strip()
    
    if not BOARD_ID:
        print("❌ 未提供看板ID，程序终止")
        exit(1)
    
    try:
        print(f"\n🎯 目标看板ID: {BOARD_ID}")
        
        # 创建分析器实例
        analyzer = TrelloOrderAnalyzer(BOARD_ID)
        
        # 测试看板访问权限
        print("\n🔐 测试看板访问权限...")
        test_url = f"https://api.trello.com/1/boards/{BOARD_ID}"
        response = requests.get(test_url, params=analyzer.params)
        
        if response.status_code == 200:
            board_info = response.json()
            print(f"✅ 可以访问看板: {board_info.get('name', '未知名称')}")
            print(f"   看板描述: {board_info.get('desc', '无描述')}")
        else:
            print(f"❌ 无法访问看板，状态码: {response.status_code}")
            print(f"   错误信息: {response.text}")
            print("   请检查：")
            print("   1. 看板ID是否正确")
            print("   2. 您是否有权限访问这个看板")
            exit(1)
        
        # 导出报表
        report_file, df = analyzer.export_order_report()
        
        print("\n" + "="*50)
        if report_file and df is not None:
            print(f"🎉 导出完成！文件保存为: {report_file}")
            print(f"📊 数据统计:")
            print(f"   总卡片数: {len(df) if not df.empty else 0}")
            
            if not df.empty:
                print("\n📋 数据预览:")
                print(df.head(10))
                
                # 显示每列的统计
                print("\n📈 数据统计:")
                for column in ["Date", "Product_ID", "Total_amount", "Phone"]:
                    if column in df.columns:
                        non_empty = df[column].astype(str).str.strip().ne("").sum()
                        print(f"   {column}: {non_empty} 笔有数据")
            else:
                print("⚠️ 导出的文件为空")
        else:
            print("❌ 导出失败")
        
    except Exception as e:
        print(f"\n❌ 发生错误: {str(e)}")
        traceback.print_exc()

Trello 订单数据导出工具
🔐 測試 API 憑證...
✅ API 憑證有效！用戶: saolan lei
   用戶名: saolanlei

🎯 目标看板ID: Nr2kR7ni

🔐 测试看板访问权限...
✅ 可以访问看板: 2025 orders
   看板描述: 

开始导出订单数据...
🔍 获取自定义字段...
   ✅ 字段 'Discount Type' 找到 3 个选项
✅ 共找到 6 个自定义字段
🔍 获取看板列表...
✅ 找到 149 个列表
🔍 获取看板卡片...
✅ 找到 1389 张卡片

📋 处理 1389 张卡片...
⏳ 已处理 10/1389 张卡片
⏳ 已处理 20/1389 张卡片
⏳ 已处理 30/1389 张卡片
⏳ 已处理 40/1389 张卡片
⏳ 已处理 50/1389 张卡片
⏳ 已处理 60/1389 张卡片
⏳ 已处理 70/1389 张卡片
⏳ 已处理 80/1389 张卡片
⏳ 已处理 90/1389 张卡片
⏳ 已处理 100/1389 张卡片
⏳ 已处理 110/1389 张卡片
⏳ 已处理 120/1389 张卡片
⏳ 已处理 130/1389 张卡片
⏳ 已处理 140/1389 张卡片
⏳ 已处理 150/1389 张卡片
⏳ 已处理 160/1389 张卡片
⏳ 已处理 170/1389 张卡片
⏳ 已处理 180/1389 张卡片
⏳ 已处理 190/1389 张卡片
⏳ 已处理 200/1389 张卡片
⏳ 已处理 210/1389 张卡片
⏳ 已处理 220/1389 张卡片
⏳ 已处理 230/1389 张卡片
⏳ 已处理 240/1389 张卡片
⏳ 已处理 250/1389 张卡片
⏳ 已处理 260/1389 张卡片
⏳ 已处理 270/1389 张卡片
⏳ 已处理 280/1389 张卡片
⏳ 已处理 290/1389 张卡片
⏳ 已处理 300/1389 张卡片
⏳ 已处理 310/1389 张卡片
⏳ 已处理 320/1389 张卡片
⏳ 已处理 330/1389 张卡片
⏳ 已处理 340/1389 张卡片
⏳ 已处理 350/1389 张卡片
⏳ 已处理 360/1389 张卡片
⏳ 已处理 370/1389 张卡片
⏳ 已处理 380/1389 张卡片
⏳ 

In [ ]:
這是移除 Card_Name / Card_ID 欄位的完整程式碼：

In [1]:
import requests
import json
import os
import re
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd
import traceback

# 加载环境变量 - 更新为新的文件路径
load_dotenv(r"C:\Users\MI\Desktop\trello_project\trello_data.env")
API_KEY = os.getenv("TRELLO_API_KEY")
TOKEN = os.getenv("TRELLO_TOKEN")

# 測試 API Key 和 Token 是否有效
def test_api_credentials():
    """測試 API Key 和 Token 是否有效"""
    print("🔐 測試 API 憑證...")
    test_url = "https://api.trello.com/1/members/me"
    params = {
        "key": API_KEY,
        "token": TOKEN
    }
    
    try:
        response = requests.get(test_url, params=params)
        if response.status_code == 200:
            user_data = response.json()
            print(f"✅ API 憑證有效！用戶: {user_data.get('fullName', '未知')}")
            print(f"   用戶名: {user_data.get('username', '未知')}")
            return True
        else:
            print(f"❌ API 憑證無效！狀態碼: {response.status_code}")
            print(f"   錯誤訊息: {response.text}")
            return False
    except Exception as e:
        print(f"❌ API 測試失敗: {str(e)}")
        return False

class TrelloOrderAnalyzer:
    def __init__(self, board_id):
        self.base_url = "https://api.trello.com/1"
        self.params = {"key": API_KEY, "token": TOKEN}
        self.board_id = board_id
        self.custom_fields_map = {}
        self.field_options = {}
        
    def get_custom_fields(self):
        """获取自定义字段定义"""
        try:
            print("🔍 获取自定义字段...")
            url = f"{self.base_url}/boards/{self.board_id}/customFields"
            response = requests.get(url, params=self.params)
                
            if response.status_code != 200:
                print(f"⚠️ 获取自定义字段失败，状态码: {response.status_code}")
                print(f"   错误信息: {response.text}")
                return {}
                
            fields = response.json()
            for field in fields:
                field_id = field["id"]
                field_name = field["name"]
                self.custom_fields_map[field_id] = field_name
                
                # 存储下拉选项
                if field["type"] == "list" and field.get("options"):
                    self.field_options[field_id] = {
                        opt["id"]: opt["value"]["text"] for opt in field["options"]
                    }
                    print(f"   ✅ 字段 '{field_name}' 找到 {len(field['options'])} 个选项")
            
            print(f"✅ 共找到 {len(self.custom_fields_map)} 个自定义字段")
            return self.custom_fields_map
            
        except Exception as e:
            print(f"❌ 获取自定义字段时出错: {str(e)}")
            return {}
    
    def clean_value(self, value):
        """数据清洗：处理特殊字符和格式"""
        if value is None:
            return ""
            
        if isinstance(value, list):
            return ",".join([self.clean_value(v) for v in value])
        
        if isinstance(value, dict):
            # 处理复杂对象
            if "text" in value:
                return self.clean_value(value["text"])
            if "value" in value:
                return self.clean_value(value["value"])
            return json.dumps(value)
        
        if isinstance(value, (int, float)):
            return value
        
        # 字符串处理
        value = str(value)
        # 替换换行符
        value = value.replace('\n', ' ').replace('\r', ' ')
        # 保留中文、英文、数字和常见标点
        value = re.sub(r'[^\w\s\u4e00-\u9fff\u3400-\u4dbf\.,，。！？：；、\-\+\(\)（）]', '', value)
            
        return value.strip()
    
    def extract_card_data(self, card, lists_dict):
        """从卡片提取订单数据"""
        try:
            # 获取列表名称
            list_id = card.get("idList", "")
            list_name = lists_dict.get(list_id, "未知列表")
            
            # 初始化所有字段（已移除 Card_ID 和 Phone）
            order_data = {
                "List_Name": list_name,  # 直接使用列表名称
                "Date": None,
                "Product_ID": None,
                "Total_amount": None
            }
            
            # 处理自定义字段
            for field_item in card.get("customFieldItems", []):
                try:
                    field_id = field_item["idCustomField"]
                    field_name = self.custom_fields_map.get(field_id)
                    
                    if not field_name:
                        continue
                        
                    # 只处理需要的字段（已移除 Phone）
                    if field_name not in ["Date", "Product_ID", "Total_amount"]:
                        continue
                        
                    value_data = field_item.get("value")
                    
                    # 处理各种字段类型
                    if isinstance(value_data, dict):
                        # 下拉字段选项
                        if "id" in value_data:
                            option_id = value_data["id"]
                            options = self.field_options.get(field_id, {})
                            value = options.get(option_id, "")
                        # 文本字段
                        elif "text" in value_data:
                            value = value_data["text"]
                        # 数字字段
                        elif "number" in value_data:
                            try:
                                value = float(value_data["number"])
                            except (TypeError, ValueError):
                                value = value_data["number"]
                        # 日期字段
                        elif "date" in value_data:
                            value = value_data["date"].split("T")[0] if value_data["date"] else ""
                        else:
                            value = ""
                    else:
                        value = value_data
                    
                    # 应用数据清洗
                    cleaned_value = self.clean_value(value)
                    order_data[field_name] = cleaned_value
                    
                except Exception as e:
                    print(f"⚠️ 处理字段 {field_name} 时出错: {str(e)}")
                    continue
            
            # 使用截止日期作为备选Date
            if card.get("due") and not order_data["Date"]:
                order_data["Date"] = card["due"].split("T")[0] if card["due"] else ""
            
            return order_data
            
        except Exception as e:
            print(f"❌ 提取卡片数据出错: {str(e)}")
            return {
                "List_Name": "未知列表",
                "Date": None,
                "Product_ID": None,
                "Total_amount": None
            }
    
    def get_board_lists(self):
        """获取看板所有列表"""
        try:
            print("🔍 获取看板列表...")
            url = f"{self.base_url}/boards/{self.board_id}/lists"
            params = {
                **self.params,
                "fields": "id,name"
            }
            
            response = requests.get(url, params=params)
            if response.status_code == 200:
                lists = response.json()
                list_dict = {lst["id"]: lst["name"] for lst in lists}
                print(f"✅ 找到 {len(list_dict)} 个列表")
                # 显示列表名称
                for list_name in list_dict.values():
                    print(f"   📍 {list_name}")
                return list_dict
            else:
                print(f"⚠️ 获取看板列表失败，状态码: {response.status_code}")
                return {}
        except Exception as e:
            print(f"❌ 获取看板列表时出错: {str(e)}")
            return {}
    
    def get_all_cards(self):
        """获取看板所有卡片"""
        try:
            print("🔍 获取看板卡片...")
            url = f"{self.base_url}/boards/{self.board_id}/cards"
            params = {
                **self.params,
                "customFieldItems": "true",
                "fields": "name,due,desc,idList"
            }
            
            response = requests.get(url, params=params)
            if response.status_code == 200:
                cards = response.json()
                print(f"✅ 找到 {len(cards)} 张卡片")
                return cards
            else:
                print(f"❌ 获取卡片失败，状态码: {response.status_code}")
                print(f"   错误信息: {response.text}")
                return []
                
        except Exception as e:
            print(f"❌ 获取卡片时出错: {str(e)}")
            return []
    
    def aggregate_by_list_name(self, df):
        """按列表名称聚合数据，将相同列表名称的行合并"""
        if df.empty:
            return df
        
        # 定义聚合函数
        aggregation_functions = {
            "Date": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan")),
            "Product_ID": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan")),
            "Total_amount": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan"))
        }
        
        # 按列表名称分组并聚合
        df_aggregated = df.groupby("List_Name", as_index=False).agg(aggregation_functions)
        
        print(f"✅ 按列表名称聚合完成：从 {len(df)} 行合并为 {len(df_aggregated)} 行")
        return df_aggregated
    
    def export_order_report(self):
        """生成订单报表"""
        print("\n" + "="*50)
        print("开始导出订单数据...")
        
        # 1. 获取自定义字段
        self.get_custom_fields()
        
        # 2. 获取列表信息
        lists = self.get_board_lists()
        if not lists:
            print("❌ 无法获取看板列表")
            return None, None
        
        # 3. 获取所有卡片
        cards = self.get_all_cards()
        
        if not cards:
            print("❌ 没有找到任何卡片，请检查：")
            print("   1. API Key 和 Token 是否正确")
            print("   2. 看板ID是否正确")
            print("   3. 您是否有权限访问这个看板")
            return None, None
        
        orders = []
        print(f"\n📋 处理 {len(cards)} 张卡片...")
        
        for i, card in enumerate(cards):
            try:
                order_data = self.extract_card_data(card, lists)
                orders.append(order_data)
                
                if (i+1) % 10 == 0 or (i+1) == len(cards):
                    print(f"⏳ 已处理 {i+1}/{len(cards)} 张卡片")
                    
            except Exception as e:
                print(f"❌ 处理卡片 {i+1} 时出错: {str(e)}")
                continue
        
        # 创建DataFrame
        df = pd.DataFrame(orders)
        
        # 定义需要的字段顺序（已移除 Card_ID 和 Phone）
        required_fields = [
            "List_Name", 
            "Date", 
            "Product_ID", 
            "Total_amount"
        ]
        
        # 确保所有字段都存在
        for field in required_fields:
            if field not in df.columns:
                df[field] = ""
        
        # 按顺序排列字段
        df = df[required_fields]
        
        # 替换NaN值为空字符串
        df = df.fillna("")
        
        # 过滤掉完全空的行
        mask = (df["Date"] != "") | (df["Product_ID"] != "") | (df["Total_amount"] != "")
        df = df[mask]
        
        if len(df) == 0:
            print("⚠️ 警告：没有提取到任何有效数据")
            print("   请检查：")
            print("   1. 卡片是否有自定义字段")
            print("   2. 自定义字段名称是否正确")
        else:
            print(f"✅ 成功提取 {len(df)} 笔订单数据")
            
            # 按列表分组统计（聚合前）
            print("\n📊 按列表统计（聚合前）:")
            list_stats = df.groupby("List_Name").size().reset_index(name='卡片数量')
            for _, row in list_stats.iterrows():
                print(f"   📍 {row['List_Name']}: {row['卡片数量']} 张卡片")
            
            # 按列表名称聚合
            df = self.aggregate_by_list_name(df)
            
            # 按列表分组统计（聚合后）
            print("\n📊 按列表统计（聚合后）:")
            list_stats_agg = df.groupby("List_Name").size().reset_index(name='列表数量')
            for _, row in list_stats_agg.iterrows():
                print(f"   📍 {row['List_Name']}: {row['列表数量']} 个列表")
        
        # 保存CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"订单明细报表_{timestamp}.csv"
        df.to_csv(filename, index=False, encoding="utf-8-sig")
        
        return filename, df

def get_board_id_from_url(board_url):
    """从Trello看板URL中提取ID"""
    patterns = [
        r"https://trello\.com/b/([a-zA-Z0-9]+)/",
        r"b/([a-zA-Z0-9]{8})",
        r"https://trello\.com/b/([a-zA-Z0-9]+)$"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, board_url)
        if match:
            return match.group(1)
    
    return None

if __name__ == "__main__":
    print("="*50)
    print("Trello 订单数据导出工具")
    print("="*50)
    
    # 测试 API 凭据
    if not test_api_credentials():
        print("\n❌ API 凭据无效，请检查 .env 文件")
        print("   确保已正确设置：")
        print(f"   1. TRELLO_API_KEY = {API_KEY}")
        print(f"   2. TRELLO_TOKEN = {TOKEN}")
        exit(1)
    
    # 获取看板ID
    BOARD_ID = "Nr2kR7ni"  # 直接使用您的看板ID
    
    # 或者手动输入
    if not BOARD_ID:
        board_url = input("\n🔍 请输入Trello看板URL: ").strip()
        BOARD_ID = get_board_id_from_url(board_url)
        
        if not BOARD_ID:
            BOARD_ID = input("🔍 请直接输入看板ID: ").strip()
    
    if not BOARD_ID:
        print("❌ 未提供看板ID，程序终止")
        exit(1)
    
    try:
        print(f"\n🎯 目标看板ID: {BOARD_ID}")
        
        # 创建分析器实例
        analyzer = TrelloOrderAnalyzer(BOARD_ID)
        
        # 测试看板访问权限
        print("\n🔐 测试看板访问权限...")
        test_url = f"https://api.trello.com/1/boards/{BOARD_ID}"
        response = requests.get(test_url, params=analyzer.params)
        
        if response.status_code == 200:
            board_info = response.json()
            print(f"✅ 可以访问看板: {board_info.get('name', '未知名称')}")
            print(f"   看板描述: {board_info.get('desc', '无描述')}")
        else:
            print(f"❌ 无法访问看板，状态码: {response.status_code}")
            print(f"   错误信息: {response.text}")
            print("   请检查：")
            print("   1. 看板ID是否正确")
            print("   2. 您是否有权限访问这个看板")
            exit(1)
        
        # 导出报表
        report_file, df = analyzer.export_order_report()
        
        print("\n" + "="*50)
        if report_file and df is not None:
            print(f"🎉 导出完成！文件保存为: {report_file}")
            print(f"📊 数据统计:")
            print(f"   总记录数: {len(df) if not df.empty else 0}")
            
            if not df.empty:
                print("\n📋 数据预览:")
                print(df.head(10))
                
                # 显示每列的统计
                print("\n📈 数据统计:")
                for column in ["List_Name", "Date", "Product_ID", "Total_amount"]:
                    if column in df.columns:
                        if column == "List_Name":
                            unique_lists = df[column].nunique()
                            print(f"   {column}: {unique_lists} 个不同列表")
                        else:
                            non_empty = df[column].astype(str).str.strip().ne("").sum()
                            print(f"   {column}: {non_empty} 笔有数据")
                
                # 显示列信息
                print("\n📋 导出字段:")
                for i, col in enumerate(df.columns, 1):
                    print(f"   {i}. {col}")
                    
                # 显示文件大小
                file_size = os.path.getsize(report_file) / 1024  # KB
                print(f"\n💾 文件大小: {file_size:.2f} KB")
                
                # 显示聚合结果示例
                print("\n🔄 聚合结果示例（相同列表名称已合并）:")
                sample_rows = min(5, len(df))
                for i in range(sample_rows):
                    row = df.iloc[i]
                    print(f"\n   📍 {row['List_Name']}:")
                    print(f"     日期: {row['Date']}")
                    print(f"     产品ID: {row['Product_ID']}")
                    print(f"     金额: {row['Total_amount']}")
            else:
                print("⚠️ 导出的文件为空")
        else:
            print("❌ 导出失败")
        
    except Exception as e:
        print(f"\n❌ 发生错误: {str(e)}")
        traceback.print_exc()

Trello 订单数据导出工具
🔐 測試 API 憑證...
❌ API 憑證無效！狀態碼: 400
   錯誤訊息: invalid token

❌ API 凭据无效，请检查 .env 文件
   确保已正确设置：
   1. TRELLO_API_KEY = None
   2. TRELLO_TOKEN = None

🎯 目标看板ID: Nr2kR7ni

🔐 测试看板访问权限...
❌ 无法访问看板，状态码: 401
   错误信息: unauthorized permission requested
   请检查：
   1. 看板ID是否正确
   2. 您是否有权限访问这个看板

开始导出订单数据...
🔍 获取自定义字段...
⚠️ 获取自定义字段失败，状态码: 401
   错误信息: unauthorized permission requested
🔍 获取看板列表...
⚠️ 获取看板列表失败，状态码: 401
❌ 无法获取看板列表

❌ 导出失败


In [11]:
import requests
import json
import os
import re
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd
import traceback
import numpy as np

# 使用正確的路徑
env_path = r"C:\Users\MI\Desktop\desktop\trello_project\trello_data.env"

if os.path.exists(env_path):
    load_dotenv(env_path)
    print(f"✅ 載入環境變數: {env_path}")
else:
    backup_path = r"C:\Users\MI\Desktop\trello_project\trello_data.env"
    if os.path.exists(backup_path):
        load_dotenv(backup_path)
        print(f"✅ 使用備用路徑: {backup_path}")
    else:
        print("❌ 找不到 .env 檔案")
        load_dotenv()

API_KEY = os.getenv("TRELLO_API_KEY")
TOKEN = os.getenv("TRELLO_TOKEN")

print(f"\n📁 當前設定:")
print(f"   TRELLO_API_KEY: {API_KEY[:10] + '...' if API_KEY else 'None'}")
print(f"   TRELLO_TOKEN: {TOKEN[:10] + '...' if TOKEN else 'None'}")

def test_api_credentials():
    """測試 API Key 和 Token 是否有效"""
    print("\n🔐 測試 API 憑證...")
    
    if not API_KEY or not TOKEN:
        print("❌ API_KEY 或 TOKEN 為空")
        return False
    
    test_url = "https://api.trello.com/1/members/me"
    params = {
        "key": API_KEY,
        "token": TOKEN
    }
    
    try:
        response = requests.get(test_url, params=params)
        if response.status_code == 200:
            user_data = response.json()
            print(f"✅ API 憑證有效！用戶: {user_data.get('fullName', '未知')}")
            return True
        else:
            print(f"❌ API 憑證無效！狀態碼: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ API 測試失敗: {str(e)}")
        return False

class TrelloOrderAnalyzer:
    def __init__(self, board_id):
        self.base_url = "https://api.trello.com/1"
        self.params = {"key": API_KEY, "token": TOKEN}
        self.board_id = board_id
        self.custom_fields_map = {}
        self.field_options = {}
        
    def get_custom_fields(self):
        """获取自定义字段定义"""
        try:
            print("🔍 获取自定义字段...")
            url = f"{self.base_url}/boards/{self.board_id}/customFields"
            response = requests.get(url, params=self.params)
                
            if response.status_code != 200:
                print(f"⚠️ 获取自定义字段失败，状态码: {response.status_code}")
                return {}
                
            fields = response.json()
            print(f"\n📋 找到 {len(fields)} 個自定義字段:")
            
            for field in fields:
                field_id = field["id"]
                field_name = field["name"]
                field_type = field["type"]
                self.custom_fields_map[field_id] = field_name
                
                print(f"   📌 {field_name} (類型: {field_type})")
                
                # 特別標記 TEL 字段
                if "TEL" in field_name or "tel" in field_name or "電話" in field_name or "手機" in field_name:
                    print(f"      📞 ** 這是電話字段 **")
                
                # 存儲下拉選項
                if field_type == "list" and field.get("options"):
                    self.field_options[field_id] = {
                        opt["id"]: opt["value"]["text"] for opt in field["options"]
                    }
                    print(f"      選項: {len(field['options'])} 個")
            
            print(f"\n✅ 共找到 {len(self.custom_fields_map)} 个自定义字段")
            return self.custom_fields_map
            
        except Exception as e:
            print(f"❌ 获取自定义字段时出错: {str(e)}")
            return {}
    
    def clean_value(self, value):
        """数据清洗：保留原始數據"""
        if value is None:
            return ""
            
        if isinstance(value, list):
            return ",".join([str(v) for v in value])
        
        if isinstance(value, dict):
            if "text" in value:
                return str(value["text"])
            if "value" in value:
                return str(value["value"])
            if "number" in value:
                return value["number"]
            return str(value)
        
        return value
    
    def process_tel_number(self, tel):
        """處理 TEL 電話號碼"""
        print(f"      📞 原始 TEL 值: {tel} (類型: {type(tel)})")
        
        if tel is None:
            print(f"      📞 TEL 為 None，設為 0")
            return "0"
        
        # 處理 Trello 字段格式
        if isinstance(tel, dict):
            if "text" in tel:
                tel = tel["text"]
                print(f"      📞 從 dict 提取 text: {tel}")
            elif "value" in tel:
                tel = tel["value"]
                print(f"      📞 從 dict 提取 value: {tel}")
            else:
                print(f"      📞 dict 格式無法識別: {tel}")
                return "0"
        
        # 轉換為字串
        tel_str = str(tel).strip()
        print(f"      📞 處理前: '{tel_str}'")
        
        # 如果是空字串
        if tel_str == "" or tel_str.lower() == "nan" or tel_str.lower() == "none":
            print(f"      📞 空值，設為 0")
            return "0"
        
        # 只保留數字
        digits_only = re.sub(r'\D', '', tel_str)
        print(f"      📞 只保留數字: '{digits_only}'")
        
        if digits_only and len(digits_only) > 0:
            print(f"      ✅ TEL 號碼: {digits_only}")
            return digits_only
        else:
            print(f"      📞 無有效數字，設為 0")
            return "0"
    
    def process_discount_rate(self, discount_value):
        """處理折扣率"""
        if discount_value is None:
            return 0.0
        
        if isinstance(discount_value, (int, float)):
            return 0.95 if discount_value == 0.95 else 0.0
        
        if isinstance(discount_value, dict):
            if "number" in discount_value:
                try:
                    val = float(discount_value["number"])
                    return 0.95 if val == 0.95 else 0.0
                except:
                    return 0.0
        
        if isinstance(discount_value, str):
            discount_str = discount_value.strip().lower()
            if discount_str in ["0.95", "0.95折", "95折", "九五折"]:
                return 0.95
        
        return 0.0
    
    def calculate_final_price(self, total_amount, discount_rate):
        """計算折扣後的最終價格"""
        try:
            if total_amount is None or total_amount == "":
                return 0.0
            
            if isinstance(total_amount, dict):
                if "number" in total_amount:
                    amount = float(total_amount["number"])
                else:
                    amount = 0.0
            else:
                amount = float(total_amount) if total_amount else 0.0
            
            final_price = amount * (1 - discount_rate)
            return round(final_price, 2)
        except:
            return 0.0
    
    def extract_card_data(self, card, lists_dict):
        """从卡片提取订单数据 - 使用 TEL 欄位"""
        try:
            list_id = card.get("idList", "")
            list_name = lists_dict.get(list_id, "未知列表")
            
            card_name = card.get("name", "")
            print(f"\n   📇 卡片: {card_name[:30]}..." if len(card_name) > 30 else f"\n   📇 卡片: {card_name}")
            
            # 初始化所有字段 - 使用 TEL 取代 Phone
            order_data = {
                "List_Name": list_name,
                "Date": "",
                "Product_ID": "",
                "Total_amount": "",
                "TEL": "0",  # 改為 TEL，預設為0
                "Discount_Rate": 0.0,
                "Final_Price": 0.0,
                "Category": ""
            }
            
            # 处理自定义字段
            custom_fields = card.get("customFieldItems", [])
            print(f"      自定義字段數量: {len(custom_fields)}")
            
            for field_item in custom_fields:
                try:
                    field_id = field_item["idCustomField"]
                    field_name = self.custom_fields_map.get(field_id, "")
                    
                    if not field_name:
                        continue
                    
                    value_data = field_item.get("value")
                    print(f"      📎 字段: {field_name}")
                    
                    # 根據字段名稱處理
                    if "日期" in field_name or field_name == "Date":
                        if isinstance(value_data, dict):
                            if "date" in value_data:
                                order_data["Date"] = value_data["date"].split("T")[0] if value_data["date"] else ""
                                print(f"         日期: {order_data['Date']}")
                            elif "text" in value_data:
                                order_data["Date"] = value_data["text"]
                                print(f"         日期: {order_data['Date']}")
                        else:
                            order_data["Date"] = str(value_data) if value_data else ""
                            print(f"         日期: {order_data['Date']}")
                    
                    elif "產品編號" in field_name or "商品編號" in field_name or field_name == "Product_ID":
                        if isinstance(value_data, dict):
                            if "text" in value_data:
                                order_data["Product_ID"] = value_data["text"]
                            else:
                                order_data["Product_ID"] = str(value_data)
                        else:
                            order_data["Product_ID"] = str(value_data) if value_data else ""
                        print(f"         產品ID: {order_data['Product_ID']}")
                    
                    elif "總金額" in field_name or "金額" in field_name or field_name == "Total_amount":
                        if isinstance(value_data, dict):
                            if "number" in value_data:
                                order_data["Total_amount"] = value_data["number"]
                            elif "text" in value_data:
                                order_data["Total_amount"] = value_data["text"]
                            else:
                                order_data["Total_amount"] = str(value_data)
                        else:
                            order_data["Total_amount"] = value_data if value_data else ""
                        print(f"         金額: {order_data['Total_amount']}")
                    
                    # 重點：抓取 TEL 欄位
                    elif "TEL" in field_name or "tel" in field_name or "電話" in field_name or "手機" in field_name or field_name == "TEL":
                        print(f"         找到 TEL 電話字段!")
                        order_data["TEL"] = self.process_tel_number(value_data)
                    
                    elif "折扣" in field_name or field_name == "Discount_Rate" or field_name == "Discount":
                        order_data["Discount_Rate"] = self.process_discount_rate(value_data)
                        print(f"         折扣率: {order_data['Discount_Rate']}")
                    
                    elif "類別" in field_name or field_name == "Category" or field_name == "分類":
                        if isinstance(value_data, dict):
                            if "text" in value_data:
                                order_data["Category"] = value_data["text"]
                            elif "id" in value_data:
                                options = self.field_options.get(field_id, {})
                                order_data["Category"] = options.get(value_data["id"], "")
                            else:
                                order_data["Category"] = str(value_data)
                        else:
                            order_data["Category"] = str(value_data) if value_data else ""
                        print(f"         類別: {order_data['Category']}")
                    
                except Exception as e:
                    print(f"      ⚠️ 处理字段出错: {str(e)}")
                    continue
            
            # 使用截止日期作为备选Date
            if card.get("due") and (not order_data["Date"] or order_data["Date"] == ""):
                order_data["Date"] = card["due"].split("T")[0] if card["due"] else ""
                print(f"      使用截止日期: {order_data['Date']}")
            
            # 確保 TEL 不為空
            if order_data["TEL"] is None or order_data["TEL"] == "":
                order_data["TEL"] = "0"
                print(f"      📞 TEL 設為預設值: 0")
            
            # 計算最終價格
            if order_data["Total_amount"] and order_data["Total_amount"] != "":
                order_data["Final_Price"] = self.calculate_final_price(
                    order_data["Total_amount"], 
                    order_data["Discount_Rate"]
                )
                print(f"      💰 最終價格: {order_data['Final_Price']}")
            
            print(f"      📞 最終 TEL: {order_data['TEL']}")
            
            return order_data
            
        except Exception as e:
            print(f"❌ 提取卡片数据出错: {str(e)}")
            return {
                "List_Name": "未知列表",
                "Date": "",
                "Product_ID": "",
                "Total_amount": "",
                "TEL": "0",  # 改為 TEL
                "Discount_Rate": 0.0,
                "Final_Price": 0.0,
                "Category": ""
            }
    
    def get_board_lists(self):
        """获取看板所有列表"""
        try:
            print("🔍 获取看板列表...")
            url = f"{self.base_url}/boards/{self.board_id}/lists"
            params = {
                **self.params,
                "fields": "id,name"
            }
            
            response = requests.get(url, params=params)
            if response.status_code == 200:
                lists = response.json()
                list_dict = {lst["id"]: lst["name"] for lst in lists}
                print(f"✅ 找到 {len(list_dict)} 个列表")
                for list_name in list_dict.values():
                    print(f"   📍 {list_name}")
                return list_dict
            else:
                print(f"⚠️ 获取看板列表失败，状态码: {response.status_code}")
                return {}
        except Exception as e:
            print(f"❌ 获取看板列表时出错: {str(e)}")
            return {}
    
    def get_all_cards(self):
        """获取看板所有卡片"""
        try:
            print("🔍 获取看板卡片...")
            url = f"{self.base_url}/boards/{self.board_id}/cards"
            params = {
                **self.params,
                "customFieldItems": "true",
                "fields": "name,due,desc,idList"
            }
            
            response = requests.get(url, params=params)
            if response.status_code == 200:
                cards = response.json()
                print(f"✅ 找到 {len(cards)} 张卡片")
                return cards
            else:
                print(f"❌ 获取卡片失败，状态码: {response.status_code}")
                return []
                
        except Exception as e:
            print(f"❌ 获取卡片时出错: {str(e)}")
            return []
    
    def aggregate_by_list_name(self, df):
        """按列表名称聚合数据 - 將相同列表名稱合併成一行"""
        if df.empty:
            return df
        
        print("\n🔄 正在按列表名稱聚合數據（合併相同列表）...")
        
        # 定義聚合函數 - 使用 TEL 取代 Phone
        aggregation_functions = {
            "Date": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != "")),
            "Product_ID": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != "")),
            "Total_amount": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != "")),
            "TEL": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != "0")),
            "Discount_Rate": lambda x: " | ".join(set(str(v) for v in x if pd.notna(v) and v != 0)),
            "Final_Price": lambda x: sum([float(v) for v in x if pd.notna(v) and v != ""]),
            "Category": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != ""))
        }
        
        # 按列表名稱分組並聚合
        df_aggregated = df.groupby("List_Name", as_index=False).agg(aggregation_functions)
        
        # 處理折扣率欄位：如果有多個折扣率，顯示最高的那個
        if "Discount_Rate" in df_aggregated.columns:
            df_aggregated["Discount_Rate"] = df_aggregated["Discount_Rate"].apply(
                lambda x: max([float(i) for i in str(x).split(" | ") if i]) if str(x) != "" else 0
            )
        
        # 四捨五入最終價格
        if "Final_Price" in df_aggregated.columns:
            df_aggregated["Final_Price"] = df_aggregated["Final_Price"].round(2)
        
        print(f"✅ 按列表名称聚合完成：从 {len(df)} 行合并为 {len(df_aggregated)} 行")
        print(f"   📍 每個列表現在只有一行資料")
        
        return df_aggregated
    
    def export_order_report(self):
        """生成订单报表 - 使用 TEL 欄位"""
        print("\n" + "="*50)
        print("开始导出订单数据...")
        
        # 1. 获取自定义字段
        self.get_custom_fields()
        
        # 2. 获取列表信息
        lists = self.get_board_lists()
        if not lists:
            print("❌ 无法获取看板列表")
            return None, None
        
        # 3. 获取所有卡片
        cards = self.get_all_cards()
        
        if not cards:
            print("❌ 没有找到任何卡片")
            return None, None
        
        orders = []
        print(f"\n📋 处理 {len(cards)} 张卡片...")
        
        for i, card in enumerate(cards):
            try:
                order_data = self.extract_card_data(card, lists)
                orders.append(order_data)
                
                if (i+1) % 10 == 0 or (i+1) == len(cards):
                    print(f"⏳ 已处理 {i+1}/{len(cards)} 张卡片")
                    
            except Exception as e:
                print(f"❌ 处理卡片 {i+1} 时出错: {str(e)}")
                continue
        
        # 创建DataFrame
        df = pd.DataFrame(orders)
        
        # 定義需要的字段順序 - 使用 TEL 取代 Phone
        required_fields = [
            "List_Name", 
            "Date", 
            "Product_ID", 
            "Total_amount",
            "TEL",  # 改為 TEL
            "Discount_Rate",
            "Final_Price",
            "Category"
        ]
        
        # 確保所有字段都存在
        for field in required_fields:
            if field not in df.columns:
                if field == "Discount_Rate" or field == "Final_Price":
                    df[field] = 0.0
                elif field == "TEL":  # 改為 TEL
                    df[field] = "0"
                elif field == "Category":
                    df[field] = ""
                else:
                    df[field] = ""
        
        # 按順序排列字段
        df = df[required_fields]
        
        # 替換NaN值
        for col in df.columns:
            if col in ["Discount_Rate", "Final_Price"]:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
            elif col == "TEL":  # 改為 TEL
                df[col] = df[col].fillna("0")
            elif col == "Category":
                df[col] = df[col].fillna("")
            else:
                df[col] = df[col].fillna("")
        
        # 過濾掉完全空的行
        mask = (df["Date"] != "") | (df["Product_ID"] != "") | (df["Total_amount"] != "")
        df = df[mask]
        
        if len(df) == 0:
            print("⚠️ 警告：没有提取到任何有效数据")
        else:
            print(f"\n✅ 成功提取 {len(df)} 笔订单数据（聚合前）")
            
            # 顯示 TEL 統計（聚合前）
            tel_count = len(df[df["TEL"] != "0"])
            print(f"\n📞 TEL 電話號碼統計（聚合前）:")
            print(f"   有電話號碼: {tel_count} 筆")
            print(f"   無電話號碼: {len(df) - tel_count} 筆")
            
            # 顯示有 TEL 的訂單
            if tel_count > 0:
                print(f"\n   📌 有 TEL 的訂單:")
                tel_orders = df[df["TEL"] != "0"].head(5)  # 只顯示前5筆
                for idx, row in tel_orders.iterrows():
                    print(f"      - {row['List_Name']}: {row['TEL']}")
            
            # 顯示類別統計（聚合前）
            if len(df[df['Category'] != '']) > 0:
                print(f"\n📋 類別統計（聚合前）:")
                category_stats = df[df['Category'] != '']['Category'].value_counts()
                for category, count in category_stats.items():
                    print(f"   {category}: {count} 筆")
            
            # 自動按列表名稱聚合
            print("\n🔄 自動按列表名稱聚合相同列表...")
            df = self.aggregate_by_list_name(df)
            
            # 顯示聚合後的統計
            print(f"\n📊 聚合後統計:")
            print(f"   總列表數: {len(df)}")
            
            # 顯示聚合後的 TEL 統計
            tel_agg_count = len(df[df["TEL"] != ""])
            print(f"\n📞 TEL 電話號碼統計（聚合後）:")
            print(f"   有電話號碼的列表: {tel_agg_count} 個")
        
        # 保存CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"订单明细报表_TEL版_{timestamp}.csv"
        df.to_csv(filename, index=False, encoding="utf-8-sig")
        
        print(f"\n📄 CSV報表已保存: {filename}")
        print(f"💾 文件大小: {os.path.getsize(filename) / 1024:.2f} KB")
        
        return filename, df

if __name__ == "__main__":
    print("="*50)
    print("Trello 订单数据导出工具 - TEL電話版")
    print("="*50)
    
    # 測試 API 憑證
    if not test_api_credentials():
        print("\n❌ API 凭据无效，程序终止")
        exit(1)
    
    # 获取看板ID
    BOARD_ID = "Nr2kR7ni"
    
    try:
        print(f"\n🎯 目标看板ID: {BOARD_ID}")
        
        # 创建分析器实例
        analyzer = TrelloOrderAnalyzer(BOARD_ID)
        
        # 测试看板访问权限
        print("\n🔐 测试看板访问权限...")
        test_url = f"https://api.trello.com/1/boards/{BOARD_ID}"
        response = requests.get(test_url, params=analyzer.params)
        
        if response.status_code == 200:
            board_info = response.json()
            print(f"✅ 可以访问看板: {board_info.get('name', '未知名称')}")
        else:
            print(f"❌ 无法访问看板，状态码: {response.status_code}")
            exit(1)
        
        # 导出报表
        report_file, df = analyzer.export_order_report()
        
        print("\n" + "="*50)
        if report_file and df is not None:
            print(f"🎉 导出完成！")
            
            if not df.empty:
                print(f"\n📋 最終報表（每個列表一行）:")
                print(df.head(10))
                
                # 顯示欄位名稱
                print(f"\n📋 导出字段:")
                for i, col in enumerate(df.columns, 1):
                    print(f"   {i}. {col}")
        else:
            print("❌ 导出失败")
        
    except Exception as e:
        print(f"\n❌ 发生错误: {str(e)}")
        traceback.print_exc()

✅ 載入環境變數: C:\Users\MI\Desktop\desktop\trello_project\trello_data.env

📁 當前設定:
   TRELLO_API_KEY: d53f3982bb...
   TRELLO_TOKEN: ATTAcda80d...
Trello 订单数据导出工具 - TEL電話版

🔐 測試 API 憑證...
✅ API 憑證有效！用戶: saolan lei

🎯 目标看板ID: Nr2kR7ni

🔐 测试看板访问权限...
✅ 可以访问看板: 2025 orders

开始导出订单数据...
🔍 获取自定义字段...

📋 找到 9 個自定義字段:
   📌 Name (類型: text)
   📌 Date (類型: date)
   📌 Phone (類型: number)
   📌 Total_amount (類型: number)
   📌 Product_ID (類型: text)
   📌 Discount Type (類型: list)
      選項: 3 個
   📌 類別 (類型: text)
   📌 Discount_Rate (類型: text)
   📌 TEL (類型: text)
      📞 ** 這是電話字段 **

✅ 共找到 9 个自定义字段
🔍 获取看板列表...
✅ 找到 143 个列表
   📍 12.3 hoi（聖誕盲合）50個
   📍 11.4 YANYEE
   📍 10.30萍萍 Shirley 寄香港 60份大花蜜糖
   📍 10.30 Fatima Tang 120份幸運曲奇+茶套裝
   📍 10.24 j 8份茶杯
   📍 10.23 Qɪ²
   📍 10.22  Ma*2❤️5⃣
   📍 10.18 Sam 寄eBuy
   📍 10.16 🦈 6个香薰包
   📍 10.20 Winwin16个盲合
   📍 10.13 gloglo寄香港
   📍 10.15 Hailey
   📍 10.13 MiRacLe🫧
   📍 10.12 Irene Lam
   📍 10.10 Katie
   📍 10.9 Chiieng 三份
   📍 10.6 Shan 35份 寄eBuy
   📍 10.6 Lei Pui Ieng 4

In [1]:
import requests
import json
import os
import re
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd
import traceback
import numpy as np
import time

# 使用正確的路徑
env_path = r"C:\Users\MI\Desktop\desktop\trello_project\trello_data.env"

if os.path.exists(env_path):
    load_dotenv(env_path)
    print(f"✅ 載入環境變數: {env_path}")
else:
    backup_path = r"C:\Users\MI\Desktop\trello_project\trello_data.env"
    if os.path.exists(backup_path):
        load_dotenv(backup_path)
        print(f"✅ 使用備用路徑: {backup_path}")
    else:
        print("❌ 找不到 .env 檔案")
        load_dotenv()

API_KEY = os.getenv("TRELLO_API_KEY")
TOKEN = os.getenv("TRELLO_TOKEN")

print(f"\n📁 當前設定:")
print(f"   TRELLO_API_KEY: {API_KEY[:10] + '...' if API_KEY else 'None'}")
print(f"   TRELLO_TOKEN: {TOKEN[:10] + '...' if TOKEN else 'None'}")

def test_api_credentials():
    """測試 API Key 和 Token 是否有效"""
    print("\n🔐 測試 API 憑證...")
    
    if not API_KEY or not TOKEN:
        print("❌ API_KEY 或 TOKEN 為空")
        return False
    
    test_url = "https://api.trello.com/1/members/me"
    params = {
        "key": API_KEY,
        "token": TOKEN
    }
    
    try:
        response = requests.get(test_url, params=params)
        if response.status_code == 200:
            user_data = response.json()
            print(f"✅ API 憑證有效！用戶: {user_data.get('fullName', '未知')}")
            return True
        else:
            print(f"❌ API 憑證無效！狀態碼: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ API 測試失敗: {str(e)}")
        return False

class TrelloOrderAnalyzer:
    def __init__(self, board_id):
        self.base_url = "https://api.trello.com/1"
        self.params = {"key": API_KEY, "token": TOKEN}
        self.board_id = board_id
        self.custom_fields_map = {}
        self.field_options = {}
        
    def get_custom_fields(self):
        """获取自定义字段定义"""
        try:
            print("🔍 获取自定义字段...")
            url = f"{self.base_url}/boards/{self.board_id}/customFields"
            response = requests.get(url, params=self.params)
                
            if response.status_code != 200:
                print(f"⚠️ 获取自定义字段失败，状态码: {response.status_code}")
                return {}
                
            fields = response.json()
            print(f"\n📋 找到 {len(fields)} 個自定義字段:")
            
            for field in fields:
                field_id = field["id"]
                field_name = field["name"]
                field_type = field["type"]
                self.custom_fields_map[field_id] = field_name
                
                print(f"   📌 {field_name} (ID: {field_id[:8]}..., 類型: {field_type})")
                
                # 存儲下拉選單選項
                if field_type == "list" and field.get("options"):
                    self.field_options[field_id] = {
                        opt["id"]: opt["value"]["text"] for opt in field["options"]
                    }
                    print(f"      選項: {len(field['options'])} 個")
                    for opt in field["options"]:
                        print(f"        - {opt['value']['text']}")
            
            print(f"\n✅ 共找到 {len(self.custom_fields_map)} 个自定义字段")
            return self.custom_fields_map
            
        except Exception as e:
            print(f"❌ 获取自定义字段时出错: {str(e)}")
            return {}
    
    def clean_value(self, value):
        """数据清洗：保留原始數據"""
        if value is None:
            return ""
            
        if isinstance(value, list):
            return ",".join([str(v) for v in value])
        
        if isinstance(value, dict):
            if "text" in value:
                return str(value["text"])
            if "value" in value:
                return str(value["value"])
            if "number" in value:
                return value["number"]
            if "date" in value:
                return value["date"]
            return str(value)
        
        return value
    
    def process_tel_number(self, tel):
        """處理 TEL 電話號碼"""
        if tel is None:
            return "0"
        
        # 處理 Trello 字段格式
        if isinstance(tel, dict):
            if "text" in tel:
                tel = tel["text"]
            elif "value" in tel:
                tel = tel["value"]
            else:
                return "0"
        
        # 轉換為字串
        tel_str = str(tel).strip()
        
        # 如果是空字串
        if tel_str == "" or tel_str.lower() == "nan" or tel_str.lower() == "none":
            return "0"
        
        # 只保留數字
        digits_only = re.sub(r'\D', '', tel_str)
        
        if digits_only and len(digits_only) > 0:
            return digits_only
        else:
            return "0"
    
    def process_discount_rate(self, discount_value, field_name=""):
        """處理折扣率 - 支援下拉選單和文本"""
        if discount_value is None:
            return 0.0
        
        # 處理下拉選單 (Discount Type)
        if isinstance(discount_value, dict) and "id" in discount_value:
            option_id = discount_value["id"]
            # 查找對應的選項文字
            for field_id, options in self.field_options.items():
                if option_id in options:
                    option_text = options[option_id]
                    # 檢查是否為 Flat_Discount:95
                    if "Flat_Discount:95" in option_text or "95" in option_text:
                        return 0.95
                    # 其他折扣都視為0
                    else:
                        return 0.0
            return 0.0
        
        # 處理文本類型的 Discount_Rate
        if isinstance(discount_value, dict):
            if "text" in discount_value:
                discount_text = discount_value["text"]
                try:
                    val = float(discount_text)
                    return 0.95 if val == 0.95 else 0.0
                except:
                    return 0.0
            elif "number" in discount_value:
                val = discount_value["number"]
                return 0.95 if val == 0.95 else 0.0
        
        # 處理直接值
        if isinstance(discount_value, (int, float)):
            return 0.95 if discount_value == 0.95 else 0.0
        
        if isinstance(discount_value, str):
            try:
                val = float(discount_value)
                return 0.95 if val == 0.95 else 0.0
            except:
                return 0.0
        
        return 0.0
    
    def calculate_final_price(self, total_amount, discount_rate):
        """計算折扣後的最終價格"""
        try:
            if total_amount is None or total_amount == "":
                return 0.0
            
            if isinstance(total_amount, dict):
                if "number" in total_amount:
                    amount = float(total_amount["number"])
                else:
                    amount = 0.0
            else:
                amount = float(total_amount) if total_amount else 0.0
            
            final_price = amount * (1 - discount_rate)
            return round(final_price, 2)
        except:
            return 0.0
    
    def get_card_custom_fields(self, card_id):
        """單獨獲取卡片的自定義字段 - 這是關鍵修正！"""
        try:
            url = f"{self.base_url}/cards/{card_id}/customFieldItems"
            response = requests.get(url, params=self.params)
            if response.status_code == 200:
                return response.json()
            else:
                return []
        except:
            return []
    
    def extract_card_data(self, card, lists_dict):
        """从卡片提取订单数据 - 修正自定義字段獲取方式"""
        try:
            card_id = card.get("id", "")
            list_id = card.get("idList", "")
            list_name = lists_dict.get(list_id, "未知列表")
            card_name = card.get("name", "")
            
            print(f"\n   📇 卡片ID: {card_id[:8]}...")
            print(f"     名稱: {card_name[:30]}..." if len(card_name) > 30 else f"     名稱: {card_name}")
            
            # 初始化所有字段
            order_data = {
                "List_Name": list_name,
                "Date": "",
                "Product_ID": "",
                "Total_amount": "",
                "TEL": "0",
                "Discount_Rate": 0.0,
                "Final_Price": 0.0,
                "類別": ""
            }
            
            # 關鍵修正：單獨獲取此卡片的自定義字段
            custom_fields = self.get_card_custom_fields(card_id)
            print(f"      自定義字段數量: {len(custom_fields)}")
            
            if len(custom_fields) == 0:
                print(f"      ⚠️ 此卡片沒有自定義字段數據！")
                return order_data
            
            for field_item in custom_fields:
                try:
                    field_id = field_item["idCustomField"]
                    field_name = self.custom_fields_map.get(field_id, "")
                    
                    if not field_name:
                        # 如果字段名稱不在映射中，先獲取一次自定義字段
                        self.get_custom_fields()
                        field_name = self.custom_fields_map.get(field_id, "未知字段")
                    
                    value_data = field_item.get("value")
                    print(f"      📎 字段: {field_name} (ID: {field_id[:8]}...)")
                    
                    # 完全匹配 Trello 的字段名稱
                    if field_name == "Date":
                        if isinstance(value_data, dict) and "date" in value_data:
                            order_data["Date"] = value_data["date"].split("T")[0] if value_data["date"] else ""
                        print(f"         日期: {order_data['Date']}")
                    
                    elif field_name == "Product_ID":
                        if isinstance(value_data, dict) and "text" in value_data:
                            order_data["Product_ID"] = value_data["text"]
                        print(f"         產品ID: {order_data['Product_ID']}")
                    
                    elif field_name == "Total_amount":
                        if isinstance(value_data, dict) and "number" in value_data:
                            order_data["Total_amount"] = value_data["number"]
                        print(f"         金額: {order_data['Total_amount']}")
                    
                    elif field_name == "TEL":
                        order_data["TEL"] = self.process_tel_number(value_data)
                        print(f"         TEL: {order_data['TEL']}")
                    
                    elif field_name == "Discount_Rate":
                        order_data["Discount_Rate"] = self.process_discount_rate(value_data, field_name)
                        print(f"         折扣率: {order_data['Discount_Rate']}")
                    
                    elif field_name == "Discount Type":
                        order_data["Discount_Rate"] = self.process_discount_rate(value_data, field_name)
                        print(f"         折扣類型: {value_data}, 折扣率: {order_data['Discount_Rate']}")
                    
                    elif field_name == "類別":
                        if isinstance(value_data, dict) and "text" in value_data:
                            order_data["類別"] = value_data["text"]
                        print(f"         類別: {order_data['類別']}")
                    
                except Exception as e:
                    print(f"      ⚠️ 处理字段出错: {str(e)}")
                    continue
            
            # 使用截止日期作为备选Date
            if card.get("due") and (not order_data["Date"] or order_data["Date"] == ""):
                order_data["Date"] = card["due"].split("T")[0] if card["due"] else ""
                print(f"      使用截止日期: {order_data['Date']}")
            
            # 確保 TEL 不為空
            if order_data["TEL"] is None or order_data["TEL"] == "":
                order_data["TEL"] = "0"
            
            # 計算最終價格
            if order_data["Total_amount"] and order_data["Total_amount"] != "":
                order_data["Final_Price"] = self.calculate_final_price(
                    order_data["Total_amount"], 
                    order_data["Discount_Rate"]
                )
            
            print(f"      📞 最終 TEL: {order_data['TEL']}")
            print(f"      🔖 最終折扣率: {order_data['Discount_Rate']}")
            print(f"      📁 最終類別: {order_data['類別']}")
            print(f"      💰 最終價格: {order_data['Final_Price']}")
            
            return order_data
            
        except Exception as e:
            print(f"❌ 提取卡片数据出错: {str(e)}")
            return {
                "List_Name": "未知列表",
                "Date": "",
                "Product_ID": "",
                "Total_amount": "",
                "TEL": "0",
                "Discount_Rate": 0.0,
                "Final_Price": 0.0,
                "類別": ""
            }
    
    def get_board_lists(self):
        """获取看板所有列表"""
        try:
            print("🔍 获取看板列表...")
            url = f"{self.base_url}/boards/{self.board_id}/lists"
            params = {
                **self.params,
                "fields": "id,name"
            }
            
            response = requests.get(url, params=params)
            if response.status_code == 200:
                lists = response.json()
                list_dict = {lst["id"]: lst["name"] for lst in lists}
                print(f"✅ 找到 {len(list_dict)} 个列表")
                for list_name in list_dict.values():
                    print(f"   📍 {list_name}")
                return list_dict
            else:
                print(f"⚠️ 获取看板列表失败，状态码: {response.status_code}")
                return {}
        except Exception as e:
            print(f"❌ 获取看板列表时出错: {str(e)}")
            return {}
    
    def get_all_cards(self):
        """获取看板所有卡片 - 修正：只獲取基本卡片信息"""
        try:
            print("🔍 获取看板卡片...")
            url = f"{self.base_url}/boards/{self.board_id}/cards"
            params = {
                **self.params,
                "fields": "id,name,due,desc,idList"  # 不請求 customFieldItems
            }
            
            response = requests.get(url, params=params)
            if response.status_code == 200:
                cards = response.json()
                print(f"✅ 找到 {len(cards)} 张卡片")
                return cards
            else:
                print(f"❌ 获取卡片失败，状态码: {response.status_code}")
                return []
                
        except Exception as e:
            print(f"❌ 获取卡片时出错: {str(e)}")
            return []
    
    def aggregate_by_list_name(self, df):
        """按列表名称聚合数据"""
        if df.empty:
            return df
        
        print("\n🔄 正在按列表名稱聚合數據...")
        
        aggregation_functions = {
            "Date": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != "")),
            "Product_ID": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != "")),
            "Total_amount": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != "")),
            "TEL": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != "0")),
            "Discount_Rate": lambda x: max([float(v) for v in x if pd.notna(v) and v != 0]) if any(v for v in x if pd.notna(v) and v != 0) else 0,
            "Final_Price": lambda x: sum([float(v) for v in x if pd.notna(v) and v != ""]),
            "類別": lambda x: " | ".join(set(str(v) for v in x if str(v) and str(v) != "nan" and str(v) != ""))
        }
        
        df_aggregated = df.groupby("List_Name", as_index=False).agg(aggregation_functions)
        
        if "Final_Price" in df_aggregated.columns:
            df_aggregated["Final_Price"] = df_aggregated["Final_Price"].round(2)
        
        print(f"✅ 按列表名称聚合完成：从 {len(df)} 行合并为 {len(df_aggregated)} 行")
        return df_aggregated
    
    def export_order_report(self):
        """生成订单报表"""
        print("\n" + "="*50)
        print("开始导出订单数据...")
        
        # 1. 获取自定义字段
        self.get_custom_fields()
        
        # 2. 获取列表信息
        lists = self.get_board_lists()
        if not lists:
            print("❌ 无法获取看板列表")
            return None, None
        
        # 3. 获取所有卡片
        cards = self.get_all_cards()
        
        if not cards:
            print("❌ 没有找到任何卡片")
            return None, None
        
        orders = []
        print(f"\n📋 处理 {len(cards)} 张卡片...")
        
        for i, card in enumerate(cards):
            try:
                # 為每張卡片單獨獲取自定義字段
                order_data = self.extract_card_data(card, lists)
                orders.append(order_data)
                
                if (i+1) % 5 == 0 or (i+1) == len(cards):
                    print(f"⏳ 已处理 {i+1}/{len(cards)} 张卡片")
                
                # 避免 API 限流
                time.sleep(0.1)
                    
            except Exception as e:
                print(f"❌ 处理卡片 {i+1} 时出错: {str(e)}")
                continue
        
        # 创建DataFrame
        df = pd.DataFrame(orders)
        
        # 定義需要的字段順序
        required_fields = [
            "List_Name", 
            "Date", 
            "Product_ID", 
            "Total_amount",
            "TEL",
            "Discount_Rate",
            "Final_Price",
            "類別"
        ]
        
        # 確保所有字段都存在
        for field in required_fields:
            if field not in df.columns:
                if field == "Discount_Rate" or field == "Final_Price":
                    df[field] = 0.0
                elif field == "TEL":
                    df[field] = "0"
                elif field == "類別":
                    df[field] = ""
                else:
                    df[field] = ""
        
        # 按順序排列字段
        df = df[required_fields]
        
        # 替換NaN值
        for col in df.columns:
            if col in ["Discount_Rate", "Final_Price"]:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
            elif col == "TEL":
                df[col] = df[col].fillna("0")
            elif col == "類別":
                df[col] = df[col].fillna("")
            else:
                df[col] = df[col].fillna("")
        
        # 過濾掉完全空的行
        mask = (df["Date"] != "") | (df["Product_ID"] != "") | (df["Total_amount"] != "")
        df = df[mask]
        
        if len(df) == 0:
            print("⚠️ 警告：没有提取到任何有效数据")
        else:
            print(f"\n✅ 成功提取 {len(df)} 笔订单数据（聚合前）")
            
            # 顯示統計
            tel_count = len(df[df["TEL"] != "0"])
            discount_count = len(df[df["Discount_Rate"] > 0])
            category_count = len(df[df["類別"] != ""])
            
            print(f"\n📊 數據統計:")
            print(f"   📞 有電話: {tel_count} 筆")
            print(f"   🔖 有折扣: {discount_count} 筆")
            print(f"   📁 有類別: {category_count} 筆")
            
            # 自動按列表名稱聚合
            print("\n🔄 自動按列表名稱聚合相同列表...")
            df = self.aggregate_by_list_name(df)
        
        # 保存CSV
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"订单明细报表_{timestamp}.csv"
        df.to_csv(filename, index=False, encoding="utf-8-sig")
        
        print(f"\n📄 CSV報表已保存: {filename}")
        print(f"💾 文件大小: {os.path.getsize(filename) / 1024:.2f} KB")
        
        return filename, df

if __name__ == "__main__":
    print("="*50)
    print("Trello 订单数据导出工具 - API修正版")
    print("="*50)
    
    # 測試 API 憑證
    if not test_api_credentials():
        print("\n❌ API 凭据无效，程序终止")
        exit(1)
    
    # 获取看板ID
    BOARD_ID = "Nr2kR7ni"
    
    try:
        print(f"\n🎯 目标看板ID: {BOARD_ID}")
        
        # 创建分析器实例
        analyzer = TrelloOrderAnalyzer(BOARD_ID)
        
        # 测试看板访问权限
        print("\n🔐 测试看板访问权限...")
        test_url = f"https://api.trello.com/1/boards/{BOARD_ID}"
        response = requests.get(test_url, params=analyzer.params)
        
        if response.status_code == 200:
            board_info = response.json()
            print(f"✅ 可以访问看板: {board_info.get('name', '未知名称')}")
        else:
            print(f"❌ 无法访问看板，状态码: {response.status_code}")
            exit(1)
        
        # 导出报表
        report_file, df = analyzer.export_order_report()
        
        print("\n" + "="*50)
        if report_file and df is not None:
            print(f"🎉 导出完成！")
            
            if not df.empty:
                print(f"\n📋 最終報表預覽:")
                print(df.head(10))
        else:
            print("❌ 导出失败")
        
    except Exception as e:
        print(f"\n❌ 发生错误: {str(e)}")
        traceback.print_exc()

✅ 載入環境變數: C:\Users\MI\Desktop\desktop\trello_project\trello_data.env

📁 當前設定:
   TRELLO_API_KEY: d53f3982bb...
   TRELLO_TOKEN: ATTAcda80d...
Trello 订单数据导出工具 - API修正版

🔐 測試 API 憑證...
✅ API 憑證有效！用戶: saolan lei

🎯 目标看板ID: Nr2kR7ni

🔐 测试看板访问权限...
✅ 可以访问看板: 2025 orders

开始导出订单数据...
🔍 获取自定义字段...

📋 找到 9 個自定義字段:
   📌 Name (ID: 6836ad2b..., 類型: text)
   📌 Date (ID: 6836ae02..., 類型: date)
   📌 Phone (ID: 6836ae62..., 類型: number)
   📌 Total_amount (ID: 6836b484..., 類型: number)
   📌 Product_ID (ID: 6841a0b5..., 類型: text)
   📌 Discount Type (ID: 68afd9d5..., 類型: list)
      選項: 3 個
        - Manual_Discount
        - No_Discount
        - Flat_Discount:95
   📌 類別 (ID: 698da4a4..., 類型: text)
   📌 Discount_Rate (ID: 698da52c..., 類型: text)
   📌 TEL (ID: 698dbf12..., 類型: text)

✅ 共找到 9 个自定义字段
🔍 获取看板列表...
✅ 找到 143 个列表
   📍 12.3 hoi（聖誕盲合）50個
   📍 11.4 YANYEE
   📍 10.30萍萍 Shirley 寄香港 60份大花蜜糖
   📍 10.30 Fatima Tang 120份幸運曲奇+茶套裝
   📍 10.24 j 8份茶杯
   📍 10.23 Qɪ²
   📍 10.22  Ma*2❤️5⃣
   📍 10.18 Sam 寄eBuy
  